In [ ]:
import jax.numpy as np
import lineax as lx
import matplotlib.pyplot as plt
import jax.random as jr
import jax

from sciml.inverse_laplacian import create_A, create_b

jax.config.update("jax_enable_x64", True)


In [ ]:
# 1D Laplacian solver using finite differences

N = 50  # number of grid points
x = np.linspace(0, 1, N)
dx = x[1] - x[0]

# Define right-hand side f(x)
def f_sin(x):
    return 10 * np.sin(np.pi * x)

# Boundary conditions
u0 = 0.0
u1 = 0.0

A = create_A(N, dx)
b_sin = create_b(f_sin, x, u0, u1)

A2 = 2 * A

result = lx.linear_solve(A, b_sin)
u_sin = result.value
result2 = lx.linear_solve(A2, b_sin)
u2 = result2.value

# Plot the solution
fig, axs = plt.subplots(1, 3, figsize=(12, 4))

# Plot the solution
axs[0].plot(x[1:-1], u_sin[1:-1], label="Numerical Solution")
axs[0].set_xlabel("x")
axs[0].set_ylabel("u(x)")
axs[0].set_title("1D Laplacian Solution")
axs[0].legend()

axs[1].plot(x[1:-1], u2[1:-1], label="Numerical Solution")
axs[1].set_xlabel("x")
axs[1].set_ylabel("u(x)")
axs[1].set_title("1D Laplacian Solution")
axs[1].legend()

# Plot the forcing function
axs[2].plot(x, f_sin(x), label="Forcing $f(x)$", color="tab:orange")
axs[2].set_xlabel("x")
axs[2].set_ylabel("f(x)")
axs[2].set_title("Forcing Function")
axs[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Define right-hand side f(x)
a1, a2 = jr.uniform(jr.PRNGKey(5), shape=(2,))
def f(x):
    return 100 * (x) * (a1 - x) * (a2 - x) * (1 - x)

b = create_b(f, x, u0, u1)
result = lx.linear_solve(A, b)
u = result.value

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
# Plot the solution
axs[0].plot(x[1:-1], u[1:-1], label="Numerical Solution")
axs[0].set_xlabel("x")
axs[0].set_ylabel("u(x)")
axs[0].set_title("1D Laplacian Solution")
axs[0].legend()

# Plot the forcing function
axs[1].plot(x, f(x), label="Forcing $f(x)$", color="tab:orange")
axs[1].set_xlabel("x")
axs[1].set_ylabel("f(x)")
axs[1].set_title("Forcing Function")
axs[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
alpha_true = 3.
A_new = alpha_true * A
result = lx.linear_solve(A_new, b)
u_new = result.value

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
# Plot the solution
axs[0].plot(x[1:-1], u_new[1:-1], label="Numerical Solution")
axs[0].set_xlabel("x")
axs[0].set_ylabel("u(x)")
axs[0].set_title("New solution")
axs[0].legend()

# Plot the forcing function
axs[1].plot(x[1:-1], u[1:-1], label="Forcing $f(x)$", color="tab:orange")
axs[1].set_xlabel("x")
axs[1].set_ylabel("f(x)")
axs[1].set_title("Standard solution")
axs[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
print(f"True alpha: {alpha_true}")

A_full = A.operator.as_matrix() / A.scalar
A_inv = np.linalg.inv(A_full)

def error_fn(alpha):
    A_test = 1/alpha * A_inv
    u_test = A_test @ b
    return np.linalg.norm((u_test - u_new))

grad_fn = jax.value_and_grad(error_fn)

alpha = 1.
error = 1.
tol = 1e-3
while error > tol:
    error, grad = grad_fn(alpha)
    alpha -= grad
    print(f"Alpha: {alpha}, Error: {error}")

print(f"Final alpha: {alpha}, Final Error: {error}")

u_test = (1/alpha) * (A_inv @ b)
fig, ax = plt.subplots(figsize=(6, 4))
# Plot the solution
ax.plot(x[1:-1], u_test[1:-1], label="Inverted Solution")
ax.plot(x[1:-1], u_new[1:-1], label="True Solution", linestyle="dashed")
ax.set_xlabel("x")
ax.set_ylabel("u(x)")
ax.set_title("Inverted vs True Solution")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Add noise to the data and show

noise_level = 0.0001
u_noisy = u_sin + noise_level * jr.normal(jr.PRNGKey(0), shape=u_sin.shape)

# Compare true with noisy data
plt.plot(x[1:-1], u_sin[1:-1], label="Numerical Solution")
plt.plot(x[1:-1], u_noisy[1:-1], label="Noisy Data", color='orange')
plt.xlabel("x")
plt.ylabel("u(x)")
plt.title("1D Laplacian Solution")
plt.legend()
plt.show()

In [ ]:
f_est = -(A.mv(u_noisy))

# Plot the estimated f(x) vs the true f(x)
plt.plot(x[1:-1], f_sin(x)[1:-1], label="True f(x)")
plt.plot(x[1:-1], f_est[1:-1], label="Estimated f(x) from noisy u", linestyle="--")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Inverse Problem: Recovering f(x)")
plt.legend()
plt.show()